# Custom checkpoint: inference + retry/fallback + eval

Runs YOUR fine-tuned checkpoint (not a stock HF Hub model) over a chosen set of tasks, loading each dataset's own `huggingface_format/<name>/test` split directly (as in the code this notebook is based on) rather than `multitask/data/eval/`. Two things added on top of that original flow:

1. **Retry, then escalate to a fallback model, on obviously-bad samples.** `vllm_structured_infer.run()` already retries a sample in isolation if its completion has a lone surrogate, fails to parse as JSON, fails `target_schema.validate()`, or looks degenerately repetitive (the literal failure this was built for -- see the "Repetition detection" markdown cell below for the real example and the false-positive measurement). If it's STILL bad after every retry, and you've configured `FALLBACK_MODEL`, it escalates to that second model for one more shot before giving up and storing whatever it produced.
2. **Evaluation runs immediately after each dataset's inference**, using that dataset's own real scorer (`inference_and_eval.eval.run_eval.run_eval`, or `inference_and_eval.eval.paragraph_eval.run_paragraph_eval` for the one dataset with no doc-level gold) -- so you get inference + score in one pass per dataset instead of a separate manual step.

**Not runnable in this sandbox** (no `vllm`, no GPU, and no access to your checkpoint paths) -- written to be copied into an environment that has both. Nothing in this notebook has been executed here; read it before running it.

In [ ]:
import sys
from pathlib import Path

sys.path.append("/path/to/meddata/")
sys.path.append("/path/to/meddata/inference_and_eval/inference/")

import contextlib
import io
import json
import re

from datasets import load_from_disk

from inference_and_eval.inference.vllm_structured_infer import _make_llm
from inference_and_eval.inference.vllm_structured_infer import run as run_inference
from inference_and_eval.eval.run_eval import run_eval
from inference_and_eval.eval.paragraph_eval import run_paragraph_eval

## Datasets

Same 11 datasets/order as the original code this is based on, plus `2010_i2b2_va`, `nli4ct`, and `MMLU-Pro-Health` as output-distribution / OOD checks (a different IE schema, a closed-label classification task, and general-health MCQA outside the clinical-note domain entirely -- to sanity-check the model isn't just overfit to the other 11 tasks' output shapes). Each entry also carries how to score it:

- `"standard"` -- most datasets: `run_eval(eval_name, completions)` reads gold from `multitask/data/eval/<eval_name>.jsonl` (or that dataset's own builder for the BRAT-family ones). `eval_name` is the same string as the HF-format directory name for all of these.
- `"paragraph"` -- **`2018_n2c2_track2_paragraph` only.** This dataset has NO `multitask/data/eval/` file at all -- it's an HF-only, additive dataset (paragraph-window training signal, see `design.md`'s paragraph-chunking section), never merged into the doc-level eval pipeline. `run_eval` would raise `FileNotFoundError` on it. Score it with `inference_and_eval.eval.paragraph_eval.run_paragraph_eval` instead, which merges per-paragraph completions back into whole-note `.ann`s (offset-shifted, ids renumbered) and scores against the *same* whole-note gold `ner_re_medication` uses -- directly comparable to that task's numbers. **The original code you shared skipped index 2 (this dataset) entirely** (`range(3, 11)` jumps straight from index 1 to 3) -- included here since it now has a real scoring path; delete its row below to go back to skipping it.

**`2010_i2b2_va` needs a much bigger `max_tokens`** (see the "Per-dataset generation settings" section below) -- its task, `concept_assertion_relation`, has gold targets up to 17645 tokens. That, combined with `2010_i2b2_va`'s own clinical-note input length, may not fit under the primary model's `max_model_len=32768` set below -- if you see truncated/empty completions for this dataset specifically, raise `max_model_len` (and/or lower `2010_i2b2_va`'s `max_tokens` override, accepting more truncation) rather than assuming it's a model-quality issue.

`nli4ct` and `MMLU-Pro-Health` are both short-answer (label / letter choice) tasks, same DEFAULT_GEN as everything else -- no override needed.

In [ ]:
root = Path("/path/to/meddata/multitask/huggingface_format/")

# (hf_format_dir_name, eval_kind, eval_name_or_None)
# eval_name is None for "paragraph" -- run_paragraph_eval doesn't take a
# dataset name, it's hardcoded to ner_re_medication_paragraph's own gold.
MAIN_DATASET_SPECS = [
    ("2018_n2c2_track1", "standard", "2018_n2c2_track1"),
    ("2018_n2c2_track2", "standard", "2018_n2c2_track2"),
    ("2018_n2c2_track2_paragraph", "paragraph", None),
    ("2019_n2c2_track1_cls", "standard", "2019_n2c2_track1_cls"),
    ("2019_n2c2_track2", "standard", "2019_n2c2_track2"),
    ("2022n2c2_track1", "standard", "2022n2c2_track1"),
    ("2022n2c2_track2_mimic", "standard", "2022n2c2_track2_mimic"),
    ("2022n2c2_track2_uw", "standard", "2022n2c2_track2_uw"),
    ("MedNLI", "standard", "MedNLI"),
    ("MedQA-USMLE-US", "standard", "MedQA-USMLE-US"),
    ("umn_abbreviation", "standard", "umn_abbreviation"),
]

# Out-of-domain / output-distribution checks -- not part of this repo's own
# 11-task training mix, added specifically to sanity-check the model isn't
# just overfit to those tasks' output shapes/domain. Add more here as they
# come up (append a (hf_format_dir_name, eval_kind, eval_name) tuple; same
# "standard"/"paragraph" eval_kind convention as MAIN_DATASET_SPECS above).
#   - 2010_i2b2_va: the other BRAT-family IE task (concept/assertion/relation
#     triples), a different entity/relation vocabulary than 2018_n2c2_track2's
#     drug-ADE schema.
#   - nli4ct: classification (entailment/contradiction), closed 2-way label set.
#   - MMLU-Pro-Health: multi-choice QA on general health knowledge -- not
#     clinical-note IE at all, same "mcqa" family as MedQA-USMLE-US but a
#     distinct source/distribution.
#   - biosses: biomedical-LITERATURE sentence-pair similarity (TAC Biomedical
#     Summarization Track sentences, SemEval-2012 0-4 scale), discretized to a
#     5-category classification (very_weak..very_strong, see
#     multitask.builders.simple_tasks.build_biosses) the same way
#     2019_n2c2_track1_cls discretizes clinical STS -- research-paper prose,
#     not clinical notes at all, the strongest OOD check here. A companion
#     candidate, EBMSASS (also literature-derived sentence-pair similarity),
#     was evaluated and NOT included: its gold annotations skew heavily
#     toward "unrelated", which would collapse most category buckets into
#     one dominant class -- a poor discriminative OOD test (confirmed the
#     opposite for biosses: real bucket counts on its 100 pairs are close to
#     balanced -- 17/9/24/25/25 across the 5 categories).
OOD_DATASET_SPECS = [
    ("2010_i2b2_va", "standard", "2010_i2b2_va"),
    ("nli4ct", "standard", "nli4ct"),
    ("MMLU-Pro-Health", "standard", "MMLU-Pro-Health"),
    ("biosses", "standard", "biosses"),
]

DATASET_SPECS = MAIN_DATASET_SPECS + OOD_DATASET_SPECS

target_datasets = [(root / name).as_posix() for name, _, _ in DATASET_SPECS]
ds_sets = [load_from_disk(path)["test"] for path in target_datasets]

print(f"{len(ds_sets)} datasets loaded ({len(MAIN_DATASET_SPECS)} main + {len(OOD_DATASET_SPECS)} OOD):")
for (name, kind, _), ds in zip(DATASET_SPECS, ds_sets):
    print(f"  [{len(ds):5d} rows] {name}  (eval: {kind})")

## Primary model

Same as your original config -- swap the commented-out checkpoint paths as needed.

In [ ]:
MODEL = "/path/to/outputs/loop_transformer/models/qwen/1_7b_std_v1/checkpoint-16165/"
# MODEL = "/path/to/outputs/loop_transformer/models/ouro/1.4b/checkpoint-25860"

BACKEND = "xgrammar"

OUTPUT_DIR = Path("/path/to/outputs/new_full_qwen")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

llm = _make_llm(
    MODEL,
    BACKEND,
    llm_kwargs={"max_model_len": 32768, "gpu_memory_utilization": 0.95},
)

print(f"loaded {MODEL} with backend={BACKEND!r}")

## Retry + fallback model

`RETRIES` re-draws a sample in isolation, up to that many times, if its completion had a lone surrogate, failed to parse as JSON, failed schema validation, or **looked degenerately repetitive** -- built specifically for the failure you hit: the same drug dict / attribute string repeated many times in one array (e.g. `"docusate sodium"`'s `Strength` list repeating `"50 mg/5 mL"` 8 times, or `"(porcine) heparin"`'s whole drug entry appearing 3 times identically).

**Detection method, and why it's not a simple compressibility check.** A general text-compressibility heuristic (gzip/zlib ratio) was tried first and *dropped after measurement*: structured clinical-IE JSON is inherently highly compressible from its schema alone (repeated key names, small closed vocabularies) -- real i2b2 gold's *median* compression ratio (0.163) was actually *lower* (more "compressible-looking") than the actual degenerate sample used to design this (0.188). No threshold separates real gold from the bad case. What's shipped instead: any list of >= 6 items where more than 60% are exact duplicates (compared by canonical JSON form, so duplicate dicts are caught the same as duplicate strings), plus a sentence-repeated-more-than-5-times check for the one plain-text task (`discharge-me`). Both were validated against every real IE task's actual gold test-split targets before shipping -- false-positive rate (legit gold flagged as "repetitive"): `ner_re_medication` 2.0%, `discharge-me` 0.2%, every other task 0.0%. A false positive here only costs one wasted retry, not a corrupted result (at `temperature=0` it reproduces the same content and falls through unchanged).

**`RETRY_TEMPERATURE` is the one setting worth reading carefully.** Your original code runs everything at `temperature=0.0` (greedy). At `temperature=0`, the sampling seed is irrelevant and a retry reproduces the *identical* broken output -- so with `RETRY_TEMPERATURE` left at `None` (defaults to `TEMPERATURE`), retries on a greedy setup would silently do nothing. Setting it to a small nonzero value (e.g. `0.3`) applies ONLY to retry/fallback attempts -- your first-pass generation stays exactly as deterministic as your original code.

**`FALLBACK_MODEL` (optional).** If a sample is still bad after every retry on the primary checkpoint, and `FALLBACK_MODEL` is set, one more attempt is made on that second model before giving up. Meant for exactly the case you described: your fine-tuned checkpoint is small and occasionally degenerates, and a bigger general-purpose instruct model can usually produce something valid even though it was never fine-tuned for this schema. Left as `None` below since I don't have your actual local model id/path for it (you mentioned something in the "Qwen 3.x 27B" range -- fill in the exact one you have) -- disabled costs nothing; the fallback is only loaded the first time some sample actually needs it, not eagerly. **Loading two models needs enough combined GPU memory for both** -- lower `gpu_memory_utilization` on each or put them on separate GPUs if you're tight on VRAM.

In [ ]:
RETRIES = 2
RETRY_TEMPERATURE = 0.3  # None = same as each call's own temperature (usually 0.0 -> no-op retries, see above)

# Fill in the exact model id/path you have locally to enable escalation, e.g.:
#   FALLBACK_MODEL = "Qwen/Qwen3-32B-Instruct"   # or your local checkpoint path
FALLBACK_MODEL = None
FALLBACK_BACKEND = None  # None = same as the dataset's own backend
FALLBACK_RETRIES = 1

fallback_llm = None
if FALLBACK_MODEL:
    fallback_llm = _make_llm(FALLBACK_MODEL, FALLBACK_BACKEND or BACKEND)
    print(f"loaded fallback model {FALLBACK_MODEL!r}")
else:
    print("no fallback model configured -- primary-only, retries fall back to strip/store as-is")

## Per-dataset generation settings

Same values as your original code: dataset 0 (`cohort_eligibility`, a tiny `{"label": "met"}`-shaped target) gets a small `max_tokens`; everything else gets your shared `max_tokens=4096` + `repetition_penalty`/`frequency_penalty` setting. `2018_n2c2_track2_paragraph` (index 2) wasn't covered by your original per-index settings since you skipped it -- given the same defaults as the "everything else" group here.

In [ ]:
# max_tokens is set explicitly per dataset below rather than left as None --
# even though vllm_structured_infer.py's own _max_tokens_for(task) would
# resolve an unset value the same way, showing the actual number here (not
# a "(task default)" placeholder) makes it obvious what will really run
# without cross-referencing that script.
#
# Values come from vllm_structured_infer.DEFAULT_MAX_TOKENS, already
# measured per TASK (tiktoken cl100k, assistant-turn gold text) and
# reviewed: 12800 for ner_re_medication (p95 2942/max 3772),
# medication_event (p95 1523/max 1675), disorder_normalization,
# family_history, sdoh_event; 32000 for concept_assertion_relation (p95
# 9953/max 17645); 12800 fallback for everything else. Re-measured directly
# against this notebook's own 15 HF test splits to confirm/extend that
# table (max token count over each dataset's real assistant-turn target,
# same methodology):
#
#   dataset                        max tokens   task (falls back to 12800 unless noted)
#   2018_n2c2_track1                        7   cohort_eligibility
#   2018_n2c2_track2                     3762   ner_re_medication          -> table default 12800 (3.4x headroom)
#   2018_n2c2_track2_paragraph           1406   ner_re_medication_paragraph (NOT in table -- own override below)
#   2019_n2c2_track1_cls                    9   sts_classification
#   2019_n2c2_track2                      791   family_history             -> table default 12800
#   2022n2c2_track1                      1675   medication_event           -> table default 12800
#   2022n2c2_track2_mimic                 722   sdoh_event                 -> table default 12800
#   2022n2c2_track2_uw                   1016   sdoh_event                 -> table default 12800
#   MedNLI / nli4ct                         8   nli
#   MedQA-USMLE-US / MMLU-Pro-Health         6   mcqa
#   umn_abbreviation                       20   abbreviation_disambiguation
#   2010_i2b2_va                        17645   concept_assertion_relation -> table default 32000 (own override below)
#   biosses                                 8   similarity_classification
#
# IMPORTANT finding #1: 2018_n2c2_track2's real max (3762) leaves only ~8%
# headroom under a flat max_tokens=4096 (the earlier version of this cell) --
# too tight (any retry/fallback-model output even slightly longer than gold
# would truncate mid-JSON and score as a total miss). Fixed by explicitly
# setting the task's own well-measured 12800 default here instead of
# guessing a number. Every classification/QA/similarity/NLI/abbreviation
# task above has a tiny real max (6-20 tokens) but NO per-task entry in
# DEFAULT_MAX_TOKENS, so it would otherwise silently inherit the 12800
# fallback too -- harmless for correctness (vLLM stops at EOS, doesn't
# generate all 12800 tokens) but needlessly generous; tightened below so a
# run-away, non-terminating generation on one of these gets capped fast
# instead of burning a much larger budget before anyone notices.
#
# IMPORTANT finding #2: 2010_i2b2_va can't actually use the script's own
# 32000 default under this notebook's max_model_len=32768 (see the primary
# model cell). vLLM's real constraint is prompt_tokens + max_tokens <=
# max_model_len -- PER REQUEST, not just on the output. 2010_i2b2_va's own
# input prompt runs up to 6585 tokens (measured, tiktoken cl100k, same
# methodology): 6585 + 32000 = 38585 > 32768, which errors on that specific
# example rather than truncating gracefully. Tightened to 24000 below:
# 6585 + 24000 = 30585, safely under 32768, while still 1.36x the real
# output max (17645) -- tighter margin than the script's own 1.8x, but the
# largest this dataset can safely take under the current max_model_len. If
# you raise max_model_len later, 32000 (the script's own default) is safe
# again.
DEFAULT_GEN = dict(
    max_tokens=12800,
    sampling_kwargs=dict(repetition_penalty=1.05, frequency_penalty=0.3, seed=1234),
)

# Only datasets that need something other than DEFAULT_GEN go here.
GEN_OVERRIDES = {
    "2018_n2c2_track1": dict(
        max_tokens=128,  # measured max 7
        sampling_kwargs=dict(repetition_penalty=1.05, seed=1234),
    ),
    # Tightened from the script's own 32000 default -- see "IMPORTANT
    # finding #2" above: prompt (up to 6585 tokens) + max_tokens must stay
    # under this notebook's max_model_len=32768.
    "2010_i2b2_va": dict(
        max_tokens=24000,
        sampling_kwargs=dict(repetition_penalty=1.05, frequency_penalty=0.3, seed=1234),
    ),
    # ner_re_medication_paragraph has no DEFAULT_MAX_TOKENS entry of its own
    # (would fall back to the 12800 default) -- tightened using its own
    # measured max (1406, 2.9x headroom) since per-paragraph outputs are far
    # shorter than the parent whole-doc task's.
    "2018_n2c2_track2_paragraph": dict(
        max_tokens=4096,
        sampling_kwargs=dict(repetition_penalty=1.05, frequency_penalty=0.3, seed=1234),
    ),
    # Tiny closed-vocabulary/short-answer tasks (measured max 6-20 tokens),
    # none with a DEFAULT_MAX_TOKENS entry of their own -- capped well above
    # their real max but far below the 12800 fallback/DEFAULT_GEN.
    **{
        name: dict(
            max_tokens=128,
            sampling_kwargs=dict(repetition_penalty=1.05, frequency_penalty=0.3, seed=1234),
        )
        for name in (
            "2019_n2c2_track1_cls", "MedNLI", "MedQA-USMLE-US", "umn_abbreviation",
            "nli4ct", "MMLU-Pro-Health", "biosses",
        )
    },
}


def gen_settings(name):
    return GEN_OVERRIDES.get(name, DEFAULT_GEN)


for name, _, _ in DATASET_SPECS:
    s = gen_settings(name)
    print(f"{name:35s} max_tokens={s['max_tokens']:6d}  sampling_kwargs={s['sampling_kwargs']}")

## Run inference + eval, per dataset

For each dataset: run inference with retry/fallback wired in, save completions to `OUTPUT_DIR`, then score immediately with that dataset's real scorer (`run_eval`, or `run_paragraph_eval` for the one paragraph dataset). Each dataset's own stdout (scorer tables, `run()`'s retry/fallback/repetition summary) is captured and printed under its own header rather than interleaved; a failure on one dataset is caught and recorded in `ERRORS` so the rest still run.

In [ ]:
completions_by_name = {}
eval_output_by_name = {}  # name -> captured stdout text (run_eval/run_paragraph_eval only print, no return value)
ERRORS = {}

for (name, eval_kind, eval_name), ds in zip(DATASET_SPECS, ds_sets):
    print("=" * 88)
    print(name)
    print("=" * 88)

    gen = gen_settings(name)
    out_path = OUTPUT_DIR / f"{name}.json"

    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            completions = run_inference(
                ds,
                MODEL,
                BACKEND,
                llm=llm,
                max_tokens=gen["max_tokens"],
                sampling_kwargs=gen["sampling_kwargs"],
                out_path=out_path.as_posix(),
                retries=RETRIES,
                retry_temperature=RETRY_TEMPERATURE,
                fallback_llm=fallback_llm,
                fallback_retries=FALLBACK_RETRIES,
            )
        completions_by_name[name] = completions
    except Exception as e:
        print(buf.getvalue())
        print(f"[inference FAILED] {name}: {e!r}")
        ERRORS[name] = ("inference", e)
        continue
    print(buf.getvalue())

    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            if eval_kind == "paragraph":
                run_paragraph_eval(completions)
            else:
                run_eval(eval_name, completions)
    except Exception as e:
        print(buf.getvalue())
        print(f"[eval FAILED] {name}: {e!r}")
        ERRORS[name] = ("eval", e)
        continue
    eval_output_by_name[name] = buf.getvalue()
    print(buf.getvalue())
    print()

## Summary

Same best-effort `_headline()` extraction as `run_inference_eval.ipynb` (JSON metrics dict, `brat_scorer`'s `F1=` lines, cohort_eligibility's/family_history's official tables, STS's Pearson/bare number) -- an index into the full output printed above, not a substitute for it. Read the actual block for any dataset whose headline is `None` or looks suspiciously like 0/1.0.

In [ ]:
def _headline(text):
    """Best-effort (metric_name, value) from one dataset's captured
    run_eval()/run_paragraph_eval() stdout, or (None, None) if nothing
    recognizable was found. Same extraction order as run_inference_eval.ipynb
    (verified against real captured output from every scorer family in this
    repo): parsed JSON's own headline-shaped keys, brat_scorer's
    combined/overall F1= line (else its first F1= line), cohort_eligibility's
    official "Overall (micro)" row, family_history's official
    "Precision/Prevision Recall F1" header + data row, STS's "Pearson" line
    or bare correlation number."""
    text = text.strip()
    if not text:
        return None, None

    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = None
    if isinstance(obj, dict):
        stack = [obj]
        while stack:
            d = stack.pop()
            for key in ("f1", "accuracy", "pearson", "exact_match", "assertion_accuracy"):
                if key in d and isinstance(d[key], (int, float)):
                    return key, d[key]
            for v in d.values():
                if isinstance(v, dict):
                    stack.append(v)

    lines = text.splitlines()
    for line in lines:
        if ("combined" in line or "overall" in line.lower()) and "F1=" in line:
            m = re.search(r"F1=([\d.]+)", line)
            if m:
                return line.split(":")[0].strip(), float(m.group(1))
    all_f1 = re.findall(r"F1=([\d.]+)", text)
    if all_f1:
        return "F1", float(all_f1[0])

    for line in lines:
        if line.strip().startswith("Overall (micro)"):
            nums = re.findall(r"[\d.]+", line)
            if len(nums) >= 2:
                return "Overall (micro) F(b=1)", float(nums[-2])
    for i, line in enumerate(lines):
        if re.search(r"(Precision|Prevision)\s+Recall\s+F", line, re.I) and i + 1 < len(lines):
            nums = re.findall(r"[\d.]+", lines[i + 1])
            if nums:
                return "F1 (last col)", float(nums[-1])
    m = re.search(r"Pearson.*?([\d.]+)\s*$", text, re.I | re.M)
    if m:
        return "Pearson r", float(m.group(1))
    last_line = lines[-1].strip() if lines else ""
    if re.fullmatch(r"[\d.]+", last_line):
        return "value", float(last_line)

    return None, None


rows = []
for name, _, _ in DATASET_SPECS:
    n = len(completions_by_name.get(name, {}))
    if name in ERRORS:
        status = "error"
    elif name in eval_output_by_name:
        status = "ok"
    else:
        status = "skipped"
    metric, value = _headline(eval_output_by_name.get(name, ""))
    rows.append({
        "dataset": name,
        "n": n,
        "status": status,
        "headline_metric": metric or "(see full output above)",
        "headline_value": value,
        "error": repr(ERRORS[name][1]) if name in ERRORS else "",
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

## Known caveats

- **Retry/fallback cost stdout, not correctness.** `run()`'s own printed summary (inside each dataset's captured block above) reports `retry_counts`, `still_bad_after_retry`, and `fallback_used_ids`/`fallback_fixed_ids` -- if a dataset's `still_bad_after_retry` list is non-empty, that dataset's completions include at least one sample that was surrogate/unparseable/schema-violating/repetitive even after every retry and (if configured) the fallback model; check `OUTPUT_DIR/<name>.json` for those ids before trusting that dataset's score.
- **Repetition detection is a targeted heuristic, not a general one** -- duplicate-list-item check (>60% exact dupes in a list of >= 6) plus a >5x-repeated-sentence check for the one plain-text task. Validated at 0.0-2.0% false-positive rate against real gold, but a genuinely correct, unusually repetitive-looking completion is still possible; a false positive here only costs one extra retry (harmless at `RETRY_TEMPERATURE` != the base temperature), not a wrong score.
- **`RETRY_TEMPERATURE` only affects retry/fallback attempts, never the first-pass generation** -- each dataset's *first* completion is exactly as deterministic as your original code (`temperature=0.0` by default inside `run()`, its own `seed` from `sampling_kwargs`). Only samples that failed a check get resampled at a different temperature/seed.
- **`FALLBACK_MODEL` is left unset above** -- fill in the exact local checkpoint/model id you have before running if you want the escalation step; with it `None`, `run()` still retries on the primary model but has nothing to escalate to, so `still_bad_after_retry` is the true final failure list.
- **`2018_n2c2_track2_paragraph` (index 2) was skipped entirely in the code this notebook is based on** -- included here since `run_paragraph_eval` gives it a real doc-level-comparable score; delete its row from `DATASET_SPECS` above to go back to skipping it.
- **`max_tokens` is a correctness parameter, not a cost knob**, for the four JSON-heavy IE tasks -- under grammar-constrained decoding, hitting the cap mid-object leaves the JSON unclosed, which fails to parse and scores as a total miss. `run()` reports truncated completions (`finish_reason == "length"`) in its own printed summary; if that count is non-zero for a dataset, raise its `max_tokens` in `GEN_OVERRIDES` before trusting that dataset's score.
- **The decoding grammar constrains JSON shape, not semantic correctness** -- entity/span `"text"` fields are unconstrained strings, and cross-field enum constraints (e.g. SDOH's `{"role","name","value"}`) aren't expressible in a plain JSON Schema. `target_schema.validate()` catches the second class after generation; violations are in each dataset's printed summary above.
- Nothing in this notebook has been executed in this sandbox (no `vllm`, no GPU, no access to the checkpoint paths). Every piece it calls (`run()`'s retry/fallback logic, `run_eval`, `run_paragraph_eval`, `_headline()`) has been exercised elsewhere (unit/stub tests, this repo's own self-checks), but this notebook's own control flow has not been run end to end against a real model.